# Pyr VTK Nuclei And Mesh Viewer

VTK/OpenGL viewer for rendering nuclei centroids with selected local Pyr meshes. This notebook reloads the screened root and annotation tables, computes volume-weighted nuclei centroids, renders nuclei as VTK glyphs with size derived from `nucleus_volume_sum`, and overlays locally stored decimated meshes.

Annotation fields are used as contextual metadata for roots already present in the CA3 annotation table, not as a complete biological classification source. VTK does not provide the Plotly hover behavior used in the browser viewer, so hover-related functionality is intentionally omitted.

## Imports and Paths

Import plotting, mesh, and path helpers, then define local data and output locations.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import trimesh
import vtk
from meshparty import trimesh_vtk


def find_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers and data."
    )


project_root = find_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from path_behavior import format_path, print_path
from vtk_render_utils import save_vtk_render_snapshot

# Set to True to show full absolute paths in user-facing notebook output.
show_full_path = False


In [2]:
# ------------------------------------------------------------
# USER SETTINGS: PATHS
# ------------------------------------------------------------
nuclei_data_dir = project_root / "data" / "nuclei"
mesh_data_dir = project_root / "data" / "meshes"
decimated_mesh_dir = mesh_data_dir / "dec"
vtk_output_dir = project_root / "vtk_images"

# ------------------------------------------------------------
# USER SETTINGS: DATASET AND MESH FILES
# ------------------------------------------------------------
screened_root_parquet_path = nuclei_data_dir / "c3_nuclei_root_features_screened_mat195.parquet"
nuclei_parquet_path = nuclei_data_dir / "c3_nuclei_v1_mat195.parquet"
annotation_csv_path = project_root / "data" / "mouse_hippocampus_ca3_cell_annotations_export.csv"

materialization_version = 195
dec_prcnt = 95
decimated_mesh_glob = f"mesh_*_mat{materialization_version}_dec{dec_prcnt}.ply"
voxel_resolution_nm = {'x': 18, 'y': 18, 'z': 45}
voxel_resolution_array_nm = np.array([
    voxel_resolution_nm['x'],
    voxel_resolution_nm['y'],
    voxel_resolution_nm['z'],
], dtype=float)

print_path("screened root Parquet", screened_root_parquet_path, project_root, show_full_path)
print_path("nuclei Parquet", nuclei_parquet_path, project_root, show_full_path)
print_path("annotation CSV", annotation_csv_path, project_root, show_full_path)
print_path("decimated mesh directory", decimated_mesh_dir, project_root, show_full_path)
print_path("VTK output directory", vtk_output_dir, project_root, show_full_path)


screened root Parquet: data\nuclei\c3_nuclei_root_features_screened_mat195.parquet
nuclei Parquet: data\nuclei\c3_nuclei_v1_mat195.parquet
annotation CSV: data\mouse_hippocampus_ca3_cell_annotations_export.csv
decimated mesh directory: data\meshes\dec
VTK output directory: vtk_images


## User Settings

Adjust mesh selection, nuclei display, mesh display, and camera/render settings here. Screenshot saving is controlled in the final save cell so the default public rerun does not write files.

In [ ]:
# ------------------------------------------------------------
# USER SETTINGS: MESH SELECTION
# ------------------------------------------------------------
mesh_root_ids = [648518346429302836, 648518346432510775, 648518346435068156, 648518346436621392, 648518346436629840, 648518346436698777, 648518346436797822, 648518346436805246, 648518346438828916, 648518346439575644, 648518346440739174, 648518346441158615, 648518346441603042, 648518346441998053, 648518346442010597, 648518346443207945, 648518346444264392, 648518346444314087, 648518346445010973, 648518346446683356, 648518346446690524, 648518346447910219, 648518346448038906, 648518346449469764, 648518346454927753, 648518346478162598]

mesh_root_id_counts = pd.Series(mesh_root_ids, dtype='object').value_counts()
duplicate_mesh_root_ids = mesh_root_id_counts[mesh_root_id_counts > 1]
print(f'total mesh root IDs: {len(mesh_root_ids)}')
print(f'unique mesh root IDs: {len(mesh_root_id_counts)}')
if duplicate_mesh_root_ids.empty:
    print('duplicate mesh root IDs: none')
else:
    print('duplicate mesh root IDs:')
    print(duplicate_mesh_root_ids.to_dict())

# ------------------------------------------------------------
# USER SETTINGS: NUCLEI DISPLAY
# ------------------------------------------------------------
show_annotated_nuclei = True
show_unlabeled_nuclei = True
nuclei_volume_size_transform = 'log'
nuclei_glyph_radius_min_nm = 250
nuclei_glyph_radius_max_nm = 1800
annotated_nuclei_color = (0.1, 0.5, 0.7)
unlabeled_nuclei_color = (1.0, 0.70, 0.0)
annotated_nuclei_opacity = 0.50
unlabeled_nuclei_opacity = 0.50
nuclei_sphere_theta_resolution = 8
nuclei_sphere_phi_resolution = 8

# ------------------------------------------------------------
# USER SETTINGS: MESH DISPLAY
# ------------------------------------------------------------
mesh_opacity = 0.8
mesh_random_color_seed = 195
# ------------------------------------------------------------
# USER SETTINGS: CAMERA AND RENDER
# ------------------------------------------------------------
camera_backoff = 850
render_window_size = (1800, 1100)

datastack = "zheng_ca3"
render_scale = 6


total mesh root IDs: 26
unique mesh root IDs: 26
duplicate mesh root IDs: none


## Load and Join Tables

Load the screened root table and the CA3 annotation CSV. Annotation overlap is used only to split roots into `annotated_neuron` and `unlabeled` display groups; `unlabeled` is not a biological cell-type label.

In [4]:
screened_root_df = pd.read_parquet(screened_root_parquet_path)
annotation_df = pd.read_csv(annotation_csv_path)

screened_root_df = screened_root_df.copy()
annotation_df = annotation_df.copy()

screened_root_df['pt_root_id'] = pd.to_numeric(
    screened_root_df['pt_root_id'],
    errors='raise',
).astype('Int64')
annotation_df['annotation_root_id'] = pd.to_numeric(
    annotation_df['Cell ID'],
    errors='raise',
).astype('Int64')

annotation_root_ids = set(annotation_df['annotation_root_id'].dropna().astype(int).tolist())
screened_root_df['population_label'] = screened_root_df['pt_root_id'].isin(annotation_root_ids).map({
    True: 'annotated_neuron',
    False: 'unlabeled',
})

annotation_fields_to_join = [
    field for field in ['type', 'subtypes', 'outputs', 'inputs']
    if field in annotation_df.columns
]
annotation_fields_for_join_df = annotation_df[['annotation_root_id'] + annotation_fields_to_join].copy()
annotation_fields_for_join_df = annotation_fields_for_join_df.rename(columns={
    'type': 'annotation_type',
    'subtypes': 'annotation_subtypes',
    'outputs': 'annotation_outputs',
    'inputs': 'annotation_inputs',
})

screened_root_with_annotation_df = screened_root_df.merge(
    annotation_fields_for_join_df,
    left_on='pt_root_id',
    right_on='annotation_root_id',
    how='left',
    validate='one_to_one',
)
if 'annotation_root_id' in screened_root_with_annotation_df.columns:
    screened_root_with_annotation_df = screened_root_with_annotation_df.drop(columns=['annotation_root_id'])

print(f'screened root rows: {len(screened_root_df)}')
print(f'joined rows: {len(screened_root_with_annotation_df)}')
print('population counts:')
print(screened_root_with_annotation_df['population_label'].value_counts().to_dict())


screened root rows: 13860
joined rows: 13860
population counts:
{'unlabeled': 11799, 'annotated_neuron': 2061}


## Compute Nuclei Centroids

Compute one volume-weighted centroid per screened root from nuclei positions when centroid columns are not already present. Voxel coordinates are later converted to nanometers using the `voxel_resolution_nm` settings.

In [5]:
centroid_voxel_columns = ['centroid_x_vox', 'centroid_y_vox', 'centroid_z_vox']
existing_centroid_column_sets = [
    ['centroid_x_vox', 'centroid_y_vox', 'centroid_z_vox'],
    ['x_centroid', 'y_centroid', 'z_centroid'],
    ['x_center', 'y_center', 'z_center'],
    ['x_mid', 'y_mid', 'z_mid'],
]

existing_centroid_columns = next(
    (columns for columns in existing_centroid_column_sets if set(columns).issubset(screened_root_with_annotation_df.columns)),
    None,
)

if existing_centroid_columns is not None:
    root_centroid_df = screened_root_with_annotation_df[['pt_root_id'] + existing_centroid_columns].copy()
    root_centroid_df = root_centroid_df.rename(columns=dict(zip(existing_centroid_columns, centroid_voxel_columns)))
    print(f'Using existing centroid columns: {existing_centroid_columns}')
else:
    nuclei_position_df = pd.read_parquet(
        nuclei_parquet_path,
        columns=['pt_root_id', 'pt_position', 'volume'],
    )
    nuclei_position_df = nuclei_position_df[nuclei_position_df['pt_root_id'] != 0].copy()
    nuclei_position_df['pt_root_id'] = pd.to_numeric(
        nuclei_position_df['pt_root_id'],
        errors='raise',
    ).astype('Int64')

    screened_root_id_set = set(screened_root_with_annotation_df['pt_root_id'].dropna().astype(int).tolist())
    nuclei_position_df = nuclei_position_df[nuclei_position_df['pt_root_id'].astype(int).isin(screened_root_id_set)].copy()

    position_xyz_df = pd.DataFrame(
        nuclei_position_df['pt_position'].tolist(),
        columns=centroid_voxel_columns,
        index=nuclei_position_df.index,
    )
    for column in centroid_voxel_columns:
        nuclei_position_df[column] = pd.to_numeric(position_xyz_df[column], errors='coerce')

    nuclei_position_df['centroid_weight'] = pd.to_numeric(
        nuclei_position_df['volume'],
        errors='coerce',
    ).fillna(0).clip(lower=0)

    for column in centroid_voxel_columns:
        nuclei_position_df[f'{column}_weighted'] = nuclei_position_df[column] * nuclei_position_df['centroid_weight']

    centroid_agg_df = (
        nuclei_position_df
        .groupby('pt_root_id')
        .agg(
            centroid_x_weighted_sum=('centroid_x_vox_weighted', 'sum'),
            centroid_y_weighted_sum=('centroid_y_vox_weighted', 'sum'),
            centroid_z_weighted_sum=('centroid_z_vox_weighted', 'sum'),
            centroid_weight_sum=('centroid_weight', 'sum'),
            centroid_x_mean=('centroid_x_vox', 'mean'),
            centroid_y_mean=('centroid_y_vox', 'mean'),
            centroid_z_mean=('centroid_z_vox', 'mean'),
            centroid_component_count=('pt_position', 'size'),
        )
        .reset_index()
    )

    for axis in ['x', 'y', 'z']:
        centroid_agg_df[f'centroid_{axis}_vox'] = np.where(
            centroid_agg_df['centroid_weight_sum'] > 0,
            centroid_agg_df[f'centroid_{axis}_weighted_sum'] / centroid_agg_df['centroid_weight_sum'],
            centroid_agg_df[f'centroid_{axis}_mean'],
        )

    root_centroid_df = centroid_agg_df[
        ['pt_root_id'] + centroid_voxel_columns + ['centroid_component_count', 'centroid_weight_sum']
    ].copy()

print('root centroid table shape:')
print(root_centroid_df.shape)


root centroid table shape:
(13860, 6)


In [6]:
spatial_overview_df = screened_root_with_annotation_df.merge(
    root_centroid_df,
    on='pt_root_id',
    how='left',
    validate='one_to_one',
)

missing_centroid_mask = spatial_overview_df[centroid_voxel_columns].isna().any(axis=1)
spatial_plot_df = spatial_overview_df[~missing_centroid_mask].copy()

for axis in ['x', 'y', 'z']:
    spatial_plot_df[f'centroid_{axis}_nm'] = (
        pd.to_numeric(spatial_plot_df[f'centroid_{axis}_vox'], errors='coerce') *
        voxel_resolution_nm[axis]
    )

print(f'roots included in VTK nuclei rendering: {len(spatial_plot_df)}')
print('population counts in VTK nuclei rendering:')
print(spatial_plot_df['population_label'].value_counts().to_dict())


roots included in VTK nuclei rendering: 13860
population counts in VTK nuclei rendering:
{'unlabeled': 11799, 'annotated_neuron': 2061}


## Nuclei Size Transform

Convert each root's `nucleus_volume_sum` into a VTK glyph radius in nanometers. The transform and minimum/maximum radius settings control marker size while preserving the underlying root table.

In [7]:
def transform_nucleus_volume_for_marker_size(values, method):
    values = pd.to_numeric(values, errors='coerce').astype(float)
    method = method.lower() if isinstance(method, str) else method

    if method is None:
        transformed_values = values
    elif method in ['log', 'log10']:
        transformed_values = np.log10(values.where(values > 0))
    elif method == 'log1p':
        transformed_values = np.log1p(values.clip(lower=0))
    elif method == 'sqrt':
        transformed_values = np.sqrt(values.clip(lower=0))
    elif method in ['cuberoot', 'cube_root']:
        transformed_values = np.cbrt(values.clip(lower=0))
    elif method == 'rank':
        transformed_values = values.rank(pct=True)
    else:
        raise ValueError(
            "nuclei_volume_size_transform must be one of "
            "None, 'log', 'log10', 'log1p', 'sqrt', 'cuberoot', or 'rank'."
        )

    return transformed_values.replace([np.inf, -np.inf], np.nan)

nuclei_size_values = transform_nucleus_volume_for_marker_size(
    spatial_plot_df['nucleus_volume_sum'],
    nuclei_volume_size_transform,
)
nuclei_size_min = nuclei_size_values.min(skipna=True)
nuclei_size_max = nuclei_size_values.max(skipna=True)

if pd.isna(nuclei_size_min) or nuclei_size_min == nuclei_size_max:
    spatial_plot_df['nuclei_glyph_radius_nm'] = (
        nuclei_glyph_radius_min_nm + nuclei_glyph_radius_max_nm
    ) / 2
else:
    spatial_plot_df['nuclei_glyph_radius_nm'] = nuclei_glyph_radius_min_nm + (
        (nuclei_size_values - nuclei_size_min) /
        (nuclei_size_max - nuclei_size_min)
    ) * (nuclei_glyph_radius_max_nm - nuclei_glyph_radius_min_nm)
    spatial_plot_df['nuclei_glyph_radius_nm'] = spatial_plot_df['nuclei_glyph_radius_nm'].fillna(
        nuclei_glyph_radius_min_nm
    )

print('nuclei glyph radius summary, nm:')
print(spatial_plot_df['nuclei_glyph_radius_nm'].describe())


nuclei glyph radius summary, nm:
count    13860.000000
mean      1137.710888
std        423.269799
min        250.000000
25%        788.048378
50%       1103.965962
75%       1501.568897
max       1800.000000
Name: nuclei_glyph_radius_nm, dtype: float64


## Mesh Manifest and Loading

Meshes are expected under `data/meshes/dec`, with filenames approximately matching `mesh_<root_id>_mat195_dec95.ply`. These decimated PLY meshes are large local/generated artifacts and may not be included in the GitHub repository.

The mesh workflow requires `trimesh`, `vtk`, and `meshparty`. Missing requested meshes are reported and skipped where possible; available meshes are loaded and overlaid with the nuclei glyphs.

In [8]:
def refresh_decimated_mesh_manifest():
    decimated_mesh_paths = sorted(decimated_mesh_dir.glob(decimated_mesh_glob))
    manifest_columns = ['mesh_path', 'mesh_filename', 'pt_root_id']
    decimated_mesh_manifest_df = pd.DataFrame(
        [
            {
                'mesh_path': mesh_path,
                'mesh_filename': mesh_path.name,
                'pt_root_id': int(mesh_path.name.split('_')[1]),
            }
            for mesh_path in decimated_mesh_paths
        ],
        columns=manifest_columns,
    )
    return decimated_mesh_paths, decimated_mesh_manifest_df

decimated_mesh_paths, decimated_mesh_manifest_df = refresh_decimated_mesh_manifest()
available_decimated_mesh_root_ids = set(decimated_mesh_manifest_df['pt_root_id'].astype(int).tolist())
present_mesh_root_ids = [int(root_id) for root_id in mesh_root_ids if int(root_id) in available_decimated_mesh_root_ids]
missing_mesh_root_ids = [int(root_id) for root_id in mesh_root_ids if int(root_id) not in available_decimated_mesh_root_ids]

print(f'mesh root IDs requested: {len(mesh_root_ids)}')
print(f'decimated meshes present: {len(present_mesh_root_ids)}')
print(present_mesh_root_ids)
print(f'decimated meshes missing: {len(missing_mesh_root_ids)}')
print(missing_mesh_root_ids)


mesh root IDs requested: 26
decimated meshes present: 26
[648518346429302836, 648518346432510775, 648518346435068156, 648518346436621392, 648518346436629840, 648518346436698777, 648518346436797822, 648518346436805246, 648518346438828916, 648518346439575644, 648518346440739174, 648518346441158615, 648518346441603042, 648518346441998053, 648518346442010597, 648518346443207945, 648518346444264392, 648518346444314087, 648518346445010973, 648518346446683356, 648518346446690524, 648518346447910219, 648518346448038906, 648518346449469764, 648518346454927753, 648518346478162598]
decimated meshes missing: 0
[]


In [9]:
mesh_manifest_by_root_id = (
    decimated_mesh_manifest_df
    .assign(pt_root_id=lambda df: df['pt_root_id'].astype(int))
    .set_index('pt_root_id')
)

mesh_dictionary = {}
mesh_load_records = []

for root_id in present_mesh_root_ids:
    mesh_path = mesh_manifest_by_root_id.loc[int(root_id), 'mesh_path']
    if isinstance(mesh_path, pd.Series):
        mesh_path = mesh_path.iloc[0]
    try:
        mesh_dictionary[f'neuro_{root_id}'] = trimesh.load_mesh(mesh_path)
        mesh_load_records.append({
            'pt_root_id': int(root_id),
            'mesh_path': str(mesh_path),
            'loaded': True,
            'vertices': len(mesh_dictionary[f'neuro_{root_id}'].vertices),
            'faces': len(mesh_dictionary[f'neuro_{root_id}'].faces),
            'error': '',
        })
    except Exception as exc:
        mesh_load_records.append({
            'pt_root_id': int(root_id),
            'mesh_path': str(mesh_path),
            'loaded': False,
            'vertices': None,
            'faces': None,
            'error': f'{type(exc).__name__}: {exc}',
        })

mesh_load_columns = ['pt_root_id', 'mesh_path', 'loaded', 'vertices', 'faces', 'error']
mesh_load_df = pd.DataFrame(mesh_load_records, columns=mesh_load_columns)
print(f'meshes loaded: {int(mesh_load_df["loaded"].sum())}')
mesh_load_display_df = mesh_load_df.copy()
if 'mesh_path' in mesh_load_display_df.columns:
    mesh_load_display_df['mesh_path'] = mesh_load_display_df['mesh_path'].map(
        lambda path: format_path(path, project_root, show_full_path) if path else path
    )
display(mesh_load_display_df)


meshes loaded: 26


,pt_root_id,mesh_path,loaded,vertices,faces,error
0,648518346429302836,data\meshes\dec\mesh_648518346429302836_mat195...,True,47814,92712,
1,648518346432510775,data\meshes\dec\mesh_648518346432510775_mat195...,True,49285,98435,
2,648518346435068156,data\meshes\dec\mesh_648518346435068156_mat195...,True,58271,113190,
3,648518346436621392,data\meshes\dec\mesh_648518346436621392_mat195...,True,43812,85131,
4,648518346436629840,data\meshes\dec\mesh_648518346436629840_mat195...,True,52013,103676,
5,648518346436698777,data\meshes\dec\mesh_648518346436698777_mat195...,True,31874,63605,
6,648518346436797822,data\meshes\dec\mesh_648518346436797822_mat195...,True,62775,125970,
7,648518346436805246,data\meshes\dec\mesh_648518346436805246_mat195...,True,31705,63015,
8,648518346438828916,data\meshes\dec\mesh_648518346438828916_mat195...,True,43464,86367,
9,648518346439575644,data\meshes\dec\mesh_648518346439575644_mat195...,True,29372,58758,


## VTK Actor Helpers

Define small VTK actor helpers for rendering each display group as scaled sphere glyphs.

In [10]:
def make_scaled_sphere_glyph_actor(points_nm, radii_nm, color, opacity):
    vtk_points = vtk.vtkPoints()
    vtk_points.SetNumberOfPoints(len(points_nm))
    for index, point in enumerate(points_nm):
        vtk_points.SetPoint(index, float(point[0]), float(point[1]), float(point[2]))

    radius_array = vtk.vtkFloatArray()
    radius_array.SetName('glyph_radius_nm')
    for radius in radii_nm:
        radius_array.InsertNextValue(float(radius))

    polydata = vtk.vtkPolyData()
    polydata.SetPoints(vtk_points)
    polydata.GetPointData().SetScalars(radius_array)

    sphere_source = vtk.vtkSphereSource()
    sphere_source.SetRadius(1.0)
    sphere_source.SetThetaResolution(nuclei_sphere_theta_resolution)
    sphere_source.SetPhiResolution(nuclei_sphere_phi_resolution)

    glyph = vtk.vtkGlyph3D()
    glyph.SetSourceConnection(sphere_source.GetOutputPort())
    glyph.SetInputData(polydata)
    glyph.SetScaleModeToScaleByScalar()
    glyph.SetScaleFactor(1.0)
    glyph.Update()

    mapper = vtk.vtkPolyDataMapper()
    mapper.SetInputConnection(glyph.GetOutputPort())
    mapper.ScalarVisibilityOff()

    actor = vtk.vtkActor()
    actor.SetMapper(mapper)
    actor.GetProperty().SetColor(color)
    actor.GetProperty().SetOpacity(opacity)
    return actor

def population_points_and_radii(population_label):
    population_df = spatial_plot_df[spatial_plot_df['population_label'] == population_label].copy()
    points_nm = population_df[['centroid_x_nm', 'centroid_y_nm', 'centroid_z_nm']].to_numpy(dtype=float)
    radii_nm = population_df['nuclei_glyph_radius_nm'].to_numpy(dtype=float)
    return population_df, points_nm, radii_nm


## Render Meshes and Nuclei

Render selected local meshes together with nuclei-centroid glyphs. Nuclei are displayed as `annotated_neuron` and `unlabeled`; `unlabeled` only means no match in the CA3 annotation CSV. After the interactive window opens, adjust the camera/view before running the save cell.

In [11]:
rng = np.random.default_rng(mesh_random_color_seed)
mesh_actor = {}
centroids = []

for mesh_key, mesh in mesh_dictionary.items():
    random_color = tuple(rng.random(3).tolist())
    mesh_actor[mesh_key] = trimesh_vtk.mesh_actor(mesh, opacity=mesh_opacity, color=random_color)
    centroids.append(mesh.centroid)

nuclei_actor = {}
if show_annotated_nuclei:
    annotated_df, annotated_points_nm, annotated_radii_nm = population_points_and_radii('annotated_neuron')
    nuclei_actor['annotated_nuclei'] = make_scaled_sphere_glyph_actor(
        annotated_points_nm,
        annotated_radii_nm,
        annotated_nuclei_color,
        annotated_nuclei_opacity,
    )
    centroids.append(annotated_points_nm.mean(axis=0))
    print(f'annotated nuclei rendered: {len(annotated_df)}')

if show_unlabeled_nuclei:
    unlabeled_df, unlabeled_points_nm, unlabeled_radii_nm = population_points_and_radii('unlabeled')
    nuclei_actor['unlabeled_nuclei'] = make_scaled_sphere_glyph_actor(
        unlabeled_points_nm,
        unlabeled_radii_nm,
        unlabeled_nuclei_color,
        unlabeled_nuclei_opacity,
    )
    centroids.append(unlabeled_points_nm.mean(axis=0))
    print(f'unlabeled nuclei rendered: {len(unlabeled_df)}')

all_actor = {}
all_actor.update(mesh_actor)
all_actor.update(nuclei_actor)

if centroids:
    global_mean_centroid = np.mean(np.vstack(centroids), axis=0)
else:
    global_mean_centroid = np.array([0, 0, 0], dtype=float)
    print('No centroids available for camera calculation.')

camera = trimesh_vtk.oriented_camera(global_mean_centroid, backoff=camera_backoff)
print(f'mesh actors: {len(mesh_actor)}')
print(f'nuclei actors: {len(nuclei_actor)}')
print(f'total actors: {len(all_actor)}')

render_actor = dict(all_actor)

trimesh_vtk.render_actors(render_actor.values(), camera=camera)


annotated nuclei rendered: 2061
unlabeled nuclei rendered: 11799
mesh actors: 26
nuclei actors: 2
total actors: 28


<vtkmodules.vtkRenderingOpenGL2.vtkOpenGLRenderer(0x00000215CF167410) at 0x0000021611817E20>

## Save Screenshot Locally

This cell reuses the existing VTK actors and the mutable camera object from the interactive render, so it can capture the final interactively adjusted view. When saving is enabled, it writes a unique PNG and appends a JSONL metadata record.

In [12]:
# ------------------------------------------------------------
# USER SETTINGS: SAVE CURRENT VIEW
# ------------------------------------------------------------
SAVE_RENDER = False

if not SAVE_RENDER:
    print('SAVE_RENDER is False; skipping PNG save and JSONL log.')
else:
    rendered_mesh_keys = [key for key in render_actor if str(key).startswith('neuro_')]
    mesh_load_success_count = int(mesh_load_df['loaded'].sum()) if not mesh_load_df.empty else 0
    mesh_load_failure_count = int((~mesh_load_df['loaded']).sum()) if not mesh_load_df.empty else 0
    annotated_nuclei_rendered = 'annotated_nuclei' in render_actor
    unlabeled_nuclei_rendered = 'unlabeled_nuclei' in render_actor
    annotated_nuclei_count = len(annotated_df) if annotated_nuclei_rendered else 0
    unlabeled_nuclei_count = len(unlabeled_df) if unlabeled_nuclei_rendered else 0
    nuclei_metadata = {
        'rendered': annotated_nuclei_rendered or unlabeled_nuclei_rendered,
        'annotated': {
            'rendered': annotated_nuclei_rendered,
            'count': annotated_nuclei_count,
        },
        'unlabeled': {
            'rendered': unlabeled_nuclei_rendered,
            'count': unlabeled_nuclei_count,
        },
        'total_count': annotated_nuclei_count + unlabeled_nuclei_count,
    }
    render_metadata = {
        'render_type': 'mesh_with_nuclei',
        'requested_mesh_count': len(mesh_root_ids),
        'present_mesh_count': len(present_mesh_root_ids),
        'loaded_mesh_count': mesh_load_success_count,
        'rendered_mesh_count': len(rendered_mesh_keys),
        'missing_mesh_count': len(missing_mesh_root_ids),
        'mesh_load_failure_count': mesh_load_failure_count,
        'show_annotated_nuclei': show_annotated_nuclei,
        'show_unlabeled_nuclei': show_unlabeled_nuclei,
        'nuclei': nuclei_metadata,
        'nuclei_provenance': {
            'nuclei_parquet_path': str(nuclei_parquet_path),
            'annotation_csv_path': str(annotation_csv_path),
            'volume_size_transform': nuclei_volume_size_transform,
            'glyph_radius_min_nm': nuclei_glyph_radius_min_nm,
            'glyph_radius_max_nm': nuclei_glyph_radius_max_nm,
            'annotated_color': annotated_nuclei_color,
            'unlabeled_color': unlabeled_nuclei_color,
            'annotated_opacity': annotated_nuclei_opacity,
            'unlabeled_opacity': unlabeled_nuclei_opacity,
            'sphere_theta_resolution': nuclei_sphere_theta_resolution,
            'sphere_phi_resolution': nuclei_sphere_phi_resolution,
        },
    }

    render_log_record = save_vtk_render_snapshot(
        actor_dict=render_actor,
        camera=camera,
        output_dir=vtk_output_dir,
        render_scale=render_scale,
        notebook_name='07_pyr_vtk_nuclei_and_mesh_viewer.ipynb',
        materialization_version=materialization_version,
        decimation_percent=dec_prcnt,
        mesh_directory=decimated_mesh_dir,
        voxel_resolution_nm=voxel_resolution_nm,
        datastack=datastack,
        metadata=render_metadata,
        spatial_features={'nuclei': nuclei_metadata},
        save_render=SAVE_RENDER,
    )
    print(f"Saved screenshot: {format_path(render_log_record['image_path'], project_root, show_full_path)}")


SAVE_RENDER is False; skipping PNG save and JSONL log.
